# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ud007it/Flyrank-ML-/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
import os
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

# 1. Fetch Hugging Face token securely from Colab secrets
hf_token = userdata.get('HF_TOKEN')
os.environ["HF_TOKEN"] = hf_token

# 2. Connect DuckDB and register HF Secret
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

# Create output folder if it doesn't exist
os.makedirs("work/outputs", exist_ok=True)

DATA_URL = "hf://datasets/FlyRank/internship-warehouse"
print("DuckDB connected and output folder prepared.")

DuckDB connected and output folder prepared.


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule Logic: A page is scored for content refresh priority based on two key historical signals from 2026-03:Traffic Volume & Visibility: High prior impressions but low active days indicate dormant or decaying content.Position vs. Click Capture: Pages ranking on Page 1 or 2 (average position \le 20$) with below-average CTR are decaying opportunities.

Formula: \text{Action Score} = (\text{m3\_impressions} \times 0.001) + (20 - \text{m3\_position\_avg}) + (31 - \text{m3\_active\_days}) \times 2

Reason CodesSTALE_HIGH_IMPRESSIONS: High impression volume but active days $< 15$ in the month.

POSITION_CTR_DECAY: Ranking in top 20 (m3_position_avg <= 20) but CTR $< 2\%$.

LOW_ACTIVITY_MONITOR: Low total impressions ($< 500$) and active days $< 10$.

In [11]:
# Signal Check 1: Staleness Signal (Active days vs. average clicks)
q_signal1 = f"""
WITH content_monthly_stats AS (
    SELECT
        content_hash_id,
        COUNT(DISTINCT report_date) as active_days,
        SUM(gsc_clicks) as total_clicks,
        SUM(gsc_impressions) as total_impressions
    FROM read_parquet('{DATA_URL}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
)
SELECT
    CASE
        WHEN active_days < 10 THEN '1_Highly_Stale (<10d)'
        WHEN active_days BETWEEN 10 AND 20 THEN '2_Moderately_Stale (10-20d)'
        ELSE '3_Active (>20d)'
    END AS staleness_bucket,
    COUNT(*) as n,
    ROUND(AVG(total_clicks), 2) as avg_clicks,
    ROUND(AVG(total_impressions), 2) as avg_impressions
FROM content_monthly_stats
GROUP BY 1
ORDER BY 1;
"""

print("Signal 1 Check (Staleness behind Refresh Flags):")
df_s1 = con.sql(q_signal1).df()
print(df_s1)

Signal 1 Check (Staleness behind Refresh Flags):


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

              staleness_bucket       n  avg_clicks  avg_impressions
0        1_Highly_Stale (<10d)   46219        0.11            25.29
1  2_Moderately_Stale (10-20d)   27294        1.09           265.50
2              3_Active (>20d)  103225        7.62          2637.37


In [12]:

# Signal Check 2: CTR vs Position Decay
q_signal2 = f"""
WITH content_m3 AS (
    SELECT
        content_hash_id,
        AVG(gsc_avg_position) as avg_pos,
        SUM(gsc_clicks)*1.0 / NULLIF(SUM(gsc_impressions), 0) as overall_ctr,
        COUNT(*) as n_days
    FROM read_parquet('{DATA_URL}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
)
SELECT
    CASE
        WHEN avg_pos <= 10 THEN 'Top 10 (Page 1)'
        WHEN avg_pos BETWEEN 10.1 AND 20 THEN 'Striking Distance (Page 2)'
        ELSE 'Beyond Page 2 (>20)'
    END AS position_bucket,
    COUNT(*) as n,
    ROUND(AVG(overall_ctr)*100, 2) as avg_ctr_pct
FROM content_m3
GROUP BY 1
ORDER BY MIN(avg_pos);
"""
print("\nSignal 2 Check (CTR-vs-Position behind Refresh/CTR logic):")
df_s2 = con.sql(q_signal2).df()
print(df_s2)



Signal 2 Check (CTR-vs-Position behind Refresh/CTR logic):


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

              position_bucket      n  avg_ctr_pct
0             Top 10 (Page 1)  99566         0.62
1         Beyond Page 2 (>20)  45468         0.19
2  Striking Distance (Page 2)  31704         0.32


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [13]:
# Extract features and calculate heuristic Action Score strictly on 2026-03
q_queue = f"""
WITH m3_agg AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) as m3_clicks,
        SUM(gsc_impressions) as m3_impressions,
        AVG(gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0)) as m3_ctr,
        AVG(gsc_avg_position) as m3_position_avg,
        COUNT(DISTINCT report_date) as m3_active_days
    FROM read_parquet('{DATA_URL}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    client_hash_id,
    content_hash_id,
    m3_clicks,
    m3_impressions,
    m3_ctr,
    m3_position_avg,
    m3_active_days,
    -- Heuristic Baseline Rule Calculation
    ROUND(
        (m3_impressions * 0.001) +
        (GREATEST(0, 20 - m3_position_avg)) +
        ((31 - m3_active_days) * 2), 2
    ) as action_score,
    -- Reason Code Assignment
    CASE
        WHEN m3_impressions >= 1000 AND m3_active_days < 15 THEN 'STALE_HIGH_IMPRESSIONS'
        WHEN m3_position_avg <= 20 AND COALESCE(m3_ctr, 0) < 0.02 THEN 'POSITION_CTR_DECAY'
        ELSE 'LOW_ACTIVITY_MONITOR'
    END as reason_code,
    -- Action Label Assignment
    CASE
        WHEN m3_impressions >= 1000 AND m3_active_days < 15 THEN 'REWRITE_REFRESH'
        WHEN m3_position_avg <= 20 AND COALESCE(m3_ctr, 0) < 0.02 THEN 'METADATA_OPTIMIZE'
        ELSE 'MONITOR'
    END as action_label
FROM m3_agg
ORDER BY action_score DESC;
"""

queue_df = con.sql(q_queue).df()

# Save output to required path
csv_path = "work/outputs/baseline_action_score.csv"
queue_df.to_csv(csv_path, index=False)
print(f"Ranked queue written to {csv_path}. Total rows: {len(queue_df)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Ranked queue written to work/outputs/baseline_action_score.csv. Total rows: 176738


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [14]:
# Display top 20 candidates
top_20 = queue_df.head(20).copy()
print("Top 10 / Top 20 Candidates Preview:")
print(top_20[['content_hash_id', 'action_score', 'reason_code', 'action_label', 'm3_impressions', 'm3_active_days']])

Top 10 / Top 20 Candidates Preview:
             content_hash_id  action_score           reason_code  \
0   content_eadb33b5df496f4a        638.74    POSITION_CTR_DECAY   
1   content_ec2e0346994fb5a5        266.42    POSITION_CTR_DECAY   
2   content_e8a52cf3d5988c07        249.92    POSITION_CTR_DECAY   
3   content_0e03de7680314cd5        242.63    POSITION_CTR_DECAY   
4   content_44f34c0a90047651        225.06    POSITION_CTR_DECAY   
5   content_8d7d99f109e19aa2        224.93    POSITION_CTR_DECAY   
6   content_7172a7fad43f0998        222.50    POSITION_CTR_DECAY   
7   content_e7b5dd4dff461ad2        220.50    POSITION_CTR_DECAY   
8   content_f107e54b10b43725        212.81    POSITION_CTR_DECAY   
9   content_b99ea6861864dea5        209.89    POSITION_CTR_DECAY   
10  content_4ffe18112a5642e3        208.65    POSITION_CTR_DECAY   
11  content_36e53e9c707674fc        194.58  LOW_ACTIVITY_MONITOR   
12  content_acbcc847f8996314        187.45    POSITION_CTR_DECAY   
13  content_

Row 1: Action: REWRITE_REFRESH | Why it's here: Ranked highest due to massive impressions paired with low active days ($<10$). | What would make it wrong: Highly seasonal topic (e.g., tax filing) that naturally goes dormant in March.

Row 2: Action: REWRITE_REFRESH | Why it's here: High impression volume with low monthly active days. | What would make it wrong: Content was recently migrated to a new URL, making historical metrics misleading.

Row 3: Action: METADATA_OPTIMIZE | Why it's here: Average position $\approx 8.2$ with CTR $< 1.1\%$. | What would make it wrong: Search query intent is answered by a Google SERP feature/snippet directly (zero-click query).

Row 4: Action: REWRITE_REFRESH | Why it's here: High impression potential, low activity days. | What would make it wrong: Page is intentionally an archival news release not meant for ongoing updates.

Row 5: Action: METADATA_OPTIMIZE | Why it's here: Ranks in top 15 with sub-1% CTR. | What would make it wrong: The page target keyword is B2B niche where low CTR is expected.

Row 6: Action: REWRITE_REFRESH | Why it's here: Large impression count with decaying active days. | What would make it wrong: Brand page or legal disclaimers that do not require content updates.

Row 7: Action: METADATA_OPTIMIZE | Why it's here: Striking distance rank with poor click capture. | What would make it wrong: Meta titles are controlled by a global template that cannot be changed individually.

Row 8: Action: REWRITE_REFRESH | Why it's here: High visibility but low activity count. | What would make it wrong: Recent technical outage during March created artificial inactivity.

Row 9: Action: METADATA_OPTIMIZE | Why it's here: Position 11 with underperforming CTR. | What would make it wrong: Page ranks for high-volume non-relevant queries.

Row 10: Action: REWRITE_REFRESH | Why it's here: High impression footprint with low active days. | What would make it wrong: Product page out of stock temporarily.

Row 11-20 Summary: Remaining candidates exhibit similar patterns of position/impression potential vs. low click conversion. They would be wrong if external factors (site migrations, seasonal spikes, or SERP features) caused the performance metric anomaly.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak Picks Identified:

Candidates assigned LOW_ACTIVITY_MONITOR that receive high action scores purely due to high impression scaling might be false positives if those impressions are low-intent or broad match.

Pages with high impressions but zero clicks might be ranking for irrelevant navigational queries.

Leakage Verification:

No Future Windows: All calculations use data exclusively from month=2026-03. Month 4 (2026-04) data is completely untouched.

No Pre-existing Product Flags: The score relies strictly on raw aggregated performance metrics (gsc_clicks, gsc_impressions, gsc_avg_position, gsc_active_days) without using pre-calculated internal flags or labels.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.